https://colab.research.google.com/github/cs221m/cs221m-course/blob/main/03_behavioral_analysis.ipynb

In [1]:
from IPython.display import clear_output

In [2]:
from datasets import load_dataset, concatenate_datasets # hf datasets

subjects = ['abstract_algebra', 'high_school_mathematics', 'college_mathematics']

N_SHOTS = 5

ds_list = []
dev_examples = {}

for subj in subjects:
    subj_ds = load_dataset('cais/mmlu', subj, split='test')
    
    subj_ds = subj_ds.map(lambda _: {'subject': subj}, batched=False) # create cleaner dict
    ds_list.append(subj_ds)
    
    subj_dev = load_dataset('cais/mmlu', subj, split='validation')
    dev_examples[subj] = list(subj_dev.select(range(min(N_SHOTS, len(subj_dev))))) # select few examples as dict
    
    print(f"    {subj}: {len(subj_ds)} test questions, {len(dev_examples[subj])} dev shots")
    
ds = concatenate_datasets(ds_list)

print(f"\nTotal: {len(ds)} test questions across {len(subjects)} subjects")
print(f'Few-shot: {N_SHOTS} dev examples per subject')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

    college_mathematics: 100 test questions, 5 dev shots

Total: 470 test questions across 3 subjects
Few-shot: 5 dev examples per subject


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_id = 'microsoft/Phi-3.5-mini-instruct'
revision = 'main'

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side='left', # not important
)
tokenizer.pad_token = tokenizer.eos_token # use <eos> as padding

model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.half,
            device_map='auto',
            revision=revision # download main branch
        ).eval() # inference mode 

clear_output()

In [4]:
LABELS = ['A', 'B', 'C', 'D']

def format_mmlu_prompt(question, choices, subject, labels=LABELS, few_shot_examples=None):
    subject_str = subject.replace("_", " ")
    prompt = f"The following are multiple choice questioons (with answers) about {subject_str}.\n\n"
    
    if few_shot_examples:
        for ex in few_shot_examples:
            prompt += ex['question'] + '\n'
            for i, choice in enumerate(ex['choices']):
                prompt += f'{labels[i]}. {choice}\n'
            prompt += f'Answer: {labels[ex['answer']]}\n\n'
            
    prompt += question + '\n'
    for i, choice in enumerate(choices):
        prompt += f'{labels[i]}. {choice}\n'
    prompt += "Answer:"
    return prompt

def predict(model, tokenizer, prompts):
    label_token_ids = [tokenizer.encode(label, add_special_tokens=False)[0] for label in LABELS]
    
    inputs = tokenizer(prompts, return_tensors='pt', padding='longest', padding_side='left').to(model.device)
    with torch.no_grad():
        logits = model(**inputs, use_cache=False, logits_to_keep=1).logits
    
    last_logits = logits[torch.arange(len(prompts)), -1]
    
    answer_logits = last_logits[:, label_token_ids]
    predictions = answer_logits.argmax(dim=-1).cpu().tolist()
    return predictions

In [5]:
example_prompt = format_mmlu_prompt(
    ds[0]['question'], ds[0]['choices'], ds[0]['subject'],
    few_shot_examples=dev_examples[ds[0]['subject']]
)
print(example_prompt)

The following are multiple choice questioons (with answers) about abstract algebra.

The cyclic subgroup of Z_24 generated by 18 has order
A. 4
B. 8
C. 12
D. 6
Answer: A

Find the order of the factor group Z_6/<3>.
A. 2
B. 3
C. 6
D. 12
Answer: B

Statement 1 | A permutation that is a product of m even permutations and n odd permutations is an even permutation if and only if n is even. Statement 2 | Every group is isomorphic to a group of permutations.
A. True, True
B. False, False
C. True, False
D. False, True
Answer: A

Find the order of the factor group (Z_4 x Z_12)/(<2> x <2>)
A. 2
B. 3
C. 4
D. 12
Answer: C

Find the maximum possible order for some element of Z_4 x Z_6.
A. 4
B. 6
C. 12
D. 24
Answer: C

Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
A. 0
B. 4
C. 2
D. 6
Answer:


In [6]:
from tqdm import tqdm
from collections import defaultdict

BATCH_SIZE = 4

prompts = [
    format_mmlu_prompt(ex['question'], ex['choices'], ex['subject'],
                        few_shot_examples=dev_examples[ex['subject']])
    for ex in ds
]
gold_labels = [ex['answer'] for ex in ds]
example_subjects = [ex['subject'] for ex in ds]

all_preds = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Evaluating'):
    batch = prompts[i: i + BATCH_SIZE]
    preds = predict(model, tokenizer, batch)
    all_preds.extend(preds)
    
    del batch, preds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
























































































































Evaluating: 100%|██████████| 118/118 [01:55<00:00,  1.02it/s]


In [7]:
correct = sum(p == g for p,g in zip(all_preds, gold_labels))
accuracy = correct / len(gold_labels)

print(f'\nOverall accuracy ({N_SHOTS}-shot): {correct}/{len(gold_labels)} = {accuracy:.1%}\n')


Overall accuracy (5-shot): 168/470 = 35.7%



In [8]:
subject_correct = defaultdict(int)
subject_total = defaultdict(int)

for p, g, s in zip(all_preds, gold_labels, example_subjects):
    subject_total[s] += 1
    if p == g:
        subject_correct[s] += 1

for s in subjects:
    acc = subject_correct[s] / subject_total[s]
    print(f'    {s}: {subject_correct[s]}/{subject_total[s]} = {acc:.1%}')

    abstract_algebra: 35/100 = 35.0%
    high_school_mathematics: 88/270 = 32.6%
    college_mathematics: 45/100 = 45.0%


In [26]:
import random

def permute_choices(choices, answer, seed=None):
    rng = random.Random(seed)
    indices = list(range(len(choices)))
    rng.shuffle(indices)
    permuted_choices = [choices[i] for i in indices]
    new_answer = indices.index(answer)
    return permuted_choices, new_answer

def manipulate_choices(choices, answer):
    indices = list(range(len(choices)))
    indices = indices[-1:] + indices[:-1]
    manipulated_choices = [choices[i] for i in indices]
    new_answer = indices.index(answer)
    return manipulated_choices, new_answer

original_choices = ds[0]['choices']
original_answers = ds[0]['answer']

permuted_choices, permuted_answers = permute_choices(original_choices, original_answers)
manipulated_choices, manipulated_answers = manipulate_choices(original_choices, original_answers)

print("Original:")
for i, c in enumerate(original_choices):
    marker = ' <-- correct' if i == original_answers else ""
    print(f'    {LABELS[i]}. {c}{marker}')
    
print("\nPermuted:")
for i, c in enumerate(permuted_choices):
    marker = ' <-- correct' if i == permuted_answers else ""
    print(f'    {LABELS[i]}. {c}{marker}')
    
print("\nManipulated:")
for i, c in enumerate(manipulated_choices):
    marker = ' <-- correct' if i == manipulated_answers else ""
    print(f'    {LABELS[i]}. {c}{marker}')

Original:
    A. 0
    B. 4 <-- correct
    C. 2
    D. 6

Permuted:
    A. 4 <-- correct
    B. 2
    C. 0
    D. 6

Manipulated:
    A. 6
    B. 0
    C. 4 <-- correct
    D. 2


In [10]:
import numpy as np

N_TRIALS = 3

trial_preds = []
trial_gold = []

for trial in range(N_TRIALS):
    perm_prompts = []
    perm_gold = []
    
    for idx, ex in enumerate(ds):
        
        seed = trial * len(ds) + idx
        perm_choices, perm_answer = permute_choices(ex['choices'], ex['answer'], seed=seed)
        
        perm_prompts.append(format_mmlu_prompt(
            ex['question'], perm_choices, ex['subject'],
            few_shot_examples=dev_examples[ex['subject']],
        ))
        perm_gold.append(perm_answer)
        
        del perm_choices, perm_answer
        
    preds = []
    for i in tqdm(range(0, len(perm_prompts), BATCH_SIZE), desc=f"Trial {trial + 1}/{N_TRIALS}"):
        batch = perm_prompts[i : i+BATCH_SIZE]
        preds.extend(predict(model, tokenizer, batch))
        
        del batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    trial_preds.append(preds)
    trial_gold.append(perm_gold)
        

Trial 3/3: 100%|██████████| 118/118 [02:04<00:00,  1.05s/it]


In [27]:
import numpy as np

N_TRIALS = 3

trial_preds = []
trial_gold = []

for trial in range(N_TRIALS):
    perm_prompts = []
    perm_gold = []
    
    for idx, ex in enumerate(ds):
        perm_choices, perm_answer = manipulate_choices(ex['choices'], ex['answer'])
        
        perm_prompts.append(format_mmlu_prompt(
            ex['question'], perm_choices, ex['subject'],
            few_shot_examples=dev_examples[ex['subject']],
        ))
        perm_gold.append(perm_answer)
        
        del perm_choices, perm_answer
        
    preds = []
    for i in tqdm(range(0, len(perm_prompts), BATCH_SIZE), desc=f"Trial {trial + 1}/{N_TRIALS}"):
        batch = perm_prompts[i : i+BATCH_SIZE]
        preds.extend(predict(model, tokenizer, batch))
        
        del batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    trial_preds.append(preds)
    trial_gold.append(perm_gold)

Trial 3/3: 100%|██████████| 118/118 [02:04<00:00,  1.05s/it]


In [28]:
trial_accs = []

for t in range(N_TRIALS):
    acc = sum(p == g for p, g in zip(trial_preds[t], trial_gold[t])) / len(ds)
    trial_accs.append(acc)
    
print(f'Original Accuracy:  {accuracy:.1%}')
print(f'Permuted Accuracy:  {np.mean(trial_accs):.1%} ± {np.std(trial_accs):.1%} (mean ± std over {N_TRIALS} trials)')
print(f'Per-trial Accuracy: {', '.join(f"{a:.1%}" for a in trial_accs)}')
print(f'Accuracy Drop:      {accuracy - np.mean(trial_accs):+.1%}\n')

Original Accuracy:  35.7%
Permuted Accuracy:  34.9% ± 0.0% (mean ± std over 3 trials)
Per-trial Accuracy: 34.9%, 34.9%, 34.9%
Accuracy Drop:      +0.9%



In [29]:
print("Per-subject comparison (original -> permuted mean ± std dev):")

for s in subjects:
    orig_acc = subject_correct[s] / subject_total[s]
    subj_mask = [i for i, es in enumerate(example_subjects) if es == s]
    subj_accs = []
    
    for t in range(N_TRIALS):
        sc = sum(trial_preds[t][i] == trial_gold[t][i] for i in subj_mask)
        subj_accs.append(sc / len(subj_mask))
    
    print(f'    {s}: {orig_acc:.1%} -> {np.mean(subj_accs):.1%} ± {np.std(subj_accs):.1%}')

Per-subject comparison (original -> permuted mean ± std dev):
    abstract_algebra: 35.0% -> 31.0% ± 0.0%
    high_school_mathematics: 32.6% -> 33.7% ± 0.0%
    college_mathematics: 45.0% -> 42.0% ± 0.0%


In [30]:
correct_counts = []
for i in range(len(ds)):
    n_correct = sum(trial_preds[t][i] == trial_gold[t][i] for t in range(N_TRIALS))
    correct_counts.append(n_correct)
    
orig_correct = [int(p == g) for p,g in zip(all_preds, gold_labels)]

print(f"\nPer-question robustness (correct on original + N={N_TRIALS} permuted trials):")
for k in range(N_TRIALS + 1, -1, -1):
    count = sum(
        (oc + cc) == k
        for oc, cc in zip(orig_correct, correct_counts)
    )
    print(f'    {k}/{N_TRIALS+1} trials correct: {count} questions')


Per-question robustness (correct on original + N=3 permuted trials):
    4/4 trials correct: 97 questions
    3/4 trials correct: 67 questions
    2/4 trials correct: 0 questions
    1/4 trials correct: 71 questions
    0/4 trials correct: 235 questions
